In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

import joblib

In [2]:
df = pd.read_csv(
    "D:\YESCAPE-AI\YESCAPE-Version2\datasets\processed\processed_dataset_v2.csv"
)

df.head()

,title,location,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,combined_text,tokens,clean_text
0,Marketing Intern,"US, NY, New York","<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,0,1,0,Other,Internship,Unknown,Unknown,Marketing,0,marketing intern we re food and we ve created ...,"['marketing', 'intern', 'food', 'created', 'gr...",marketing intern food created groundbreaking a...
1,Customer Service - Cloud Video Production,"NZ, , Auckland","<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,0,1,0,Full-time,Not Applicable,Unknown,Marketing and Advertising,Customer Service,0,customer service cloud video production second...,"['customer', 'service', 'cloud', 'video', 'pro...",customer service cloud video production second...
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,0,1,0,Unknown,Unknown,Unknown,Unknown,Unknown,0,commissioning machinery assistant cma valor se...,"['commissioning', 'machinery', 'assistant', 'c...",commissioning machinery assistant cma valor se...
3,Account Executive - Washington DC,"US, DC, Washington",<p>Our passion for improving quality of life t...,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,<p>Our culture is anything but corporate—we ha...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,account executive washington dc our passion fo...,"['account', 'executive', 'washington', 'dc', '...",account executive washington dc passion improv...
4,Bill Review Manager,"US, FL, Fort Worth",<p>SpotSource Solutions LLC is a Global Human ...,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,<p>Full Benefits Offered</p>,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,bill review manager spotsource solutions llc i...,"['bill', 'review', 'manager', 'spotsource', 's...",bill review manager spotsource solution llc gl...


In [4]:
vectorizer = joblib.load(
    r"D:\YESCAPE-AI\YESCAPE-Version2\models\vectorizers\tfidf_vectorizer.pkl"
)

print("Vectorizer Loaded Successfully")

Vectorizer Loaded Successfully


In [5]:
X = vectorizer.transform(df["clean_text"])

y = df["fraudulent"]

print(X.shape)

print(y.shape)

(17645, 5000)
(17645,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (14116, 5000)
Testing : (3529, 5000)


In [7]:
from xgboost import XGBClassifier

In [8]:
xgb_model = XGBClassifier(

    objective="binary:logistic",

    n_estimators=100,

    learning_rate=0.1,

    max_depth=6,

    random_state=42,

    eval_metric="logloss"

)

print(xgb_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)


In [9]:
xgb_model.fit(

    X_train,

    y_train

)

print("Training Completed Successfully!")

Training Completed Successfully!


In [10]:
y_pred = xgb_model.predict(X_test)

y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Prediction Completed!")

Prediction Completed!


In [11]:
from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score

)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")

print(f"Precision: {precision:.4f}")

print(f"Recall   : {recall:.4f}")

print(f"F1 Score : {f1:.4f}")

print(f"ROC AUC  : {roc:.4f}")

Accuracy : 0.9796
Precision: 0.9386
Recall   : 0.6221
F1 Score : 0.7483
ROC AUC  : 0.9773


In [12]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[3350    7]
 [  65  107]]


In [13]:
from sklearn.metrics import classification_report

print(

    classification_report(

        y_test,

        y_pred

    )

)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3357
           1       0.94      0.62      0.75       172

    accuracy                           0.98      3529
   macro avg       0.96      0.81      0.87      3529
weighted avg       0.98      0.98      0.98      3529



In [14]:
import joblib

joblib.dump(

    xgb_model,

    "../../models/trained_models/xgboost_baseline.pkl"

)

print("Model Saved Successfully!")

Model Saved Successfully!


In [15]:
feature_importance = pd.DataFrame({

    "feature": vectorizer.get_feature_names_out(),

    "importance": xgb_model.feature_importances_

})

feature_importance = feature_importance.sort_values(

    by="importance",

    ascending=False

)

feature_importance.head(20)

,feature,importance
682,cash,0.048219
1564,entry,0.033888
3047,oil,0.025573
1509,encouraged,0.025244
102,administrative,0.023259
1887,free,0.022973
1720,facilitating,0.019405
3525,property,0.019341
2807,menu,0.014770
790,clerk,0.013719


In [16]:
feature_importance.to_csv(

    "../reports/top_feature_importance.csv",

    index=False

)

print("Feature Importance Saved Successfully!")

Feature Importance Saved Successfully!
